# KernelKG-GPT — Stage 1+2 on Colab with a **local Qwen2.5-7B (vLLM offline)**

KG-GPT-style retrieval (claim segmentation → candidate relations → top-K
ranking → triple pool) with Qwen hosted **locally** — no API, no rate limits.

Uses the **vLLM offline engine** (proven recipe: `vllm==0.7.3`,
`transformers==4.48.3`, full-precision fp16 weights, `enforce_eager`,
`enable_prefix_caching`).
A dynamic **micro-batcher** turns the concurrent per-claim calls into batched
`llm.generate([...])` calls; because every prompt shares the same long
few-shot prefix, **prefix caching** makes this very fast.

| BACKEND | Notes |
|---|---|
| **`vllm`** (default) | Fast. Pinned versions known to work (incl. T4 / A100). |
| `transformers` | Fallback using Colab's native stack + a HF micro-batcher. Slower but no version pinning. |

## Before you run
1. **Runtime → GPU** (A100 recommended — full-precision 7B fits its 80 GB).
2. **Upload KG-GPT data to Drive**, e.g. `MyDrive/kggpt_data/` with
   `factkg_train.pickle`, `factkg_dev.pickle`, `factkg_test.pickle`,
   `dbpedia_2015_undirected_light.pickle`, `relations_for_final.pickle`.
3. Run top-to-bottom. If the install asks to **Restart runtime**, do it, then
   re-run from the *Config* cell. Cache → `MyDrive/kernelkg_gpt/cache/`.


## 0. Check GPU

In [ ]:
!nvidia-smi

## 1. Configuration

In [ ]:
import os, glob, pickle, time, re, json, random
import numpy as np

# vLLM engine env (must be set before importing vllm) -- from the proven recipe
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_ATTENTION_BACKEND"] = "XFORMERS"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ---------------- Backend ----------------
BACKEND = "vllm"              # "vllm" (default) | "transformers" (fallback)
LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"   # 3B: faster + leaves lots of KV-cache room
DISABLE_THINKING = False      # Qwen2.5-Instruct is not a thinking model
LLM_TEMPERATURE = 0.0         # greedy -> deterministic, easiest to parse
LLM_MAX_TOKENS = 256          # outputs are short (a list / a few lines)
LLM_MAX_WORKERS = 256         # concurrent claims -> fills the batch queue
TOP_K = 5
MAX_TRIPLES = 30
MAX_MODEL_LEN = 4096          # vLLM context window (input + output)

# micro-batcher (both backends)
TFM_BATCH = 128               # claims per generate() call
TFM_COALESCE_WAIT = 0.05      # seconds to collect a batch

# vLLM engine knobs — tuned for 3B on a 40GB GPU with spare system RAM
VLLM_GPU_UTIL = 0.90          # ~36GB reserved; 3B weights ~6GB -> ~30GB KV cache
VLLM_MAX_NUM_SEQS = 200       # many concurrent seqs fit with the small model

# ---------------- How many claims per split ----------------
# >0 = first N ; 0 = skip ; -1 = ENTIRE split.
TRAIN_LIMIT = -1
DEV_LIMIT   = 2000
TEST_LIMIT  = -1

# ---------------- Paths (Google Drive => persistent). Dirs made after mount. --
WORK = "/content/drive/MyDrive/kernelkg_gpt"
CACHE_DIR = os.path.join(WORK, "cache")
DATA_DIRS = [
    "/content/drive/MyDrive/kggpt_data",
    "/content/drive/MyDrive",
    "/content",
]
print("Config loaded. BACKEND =", BACKEND, "| Model =", LLM_MODEL)

## 2. Install dependencies

`vllm` backend installs the **pinned, proven** combo (`vllm==0.7.3`,
`transformers==4.48.3`, `tokenizers==0.21.0`) and removes a broken `torchcodec`.
Because this changes `torch`/`transformers`, the kernel **must be restarted**
before importing them — otherwise you get errors like
`cannot import name 'is_offline_mode'` (a half-old / half-new transformers).

**This cell auto-restarts the runtime after installing.** When it restarts,
just run all cells again (Runtime → Run all): the install is idempotent, sees
the correct versions, skips, and continues without restarting.


In [ ]:
import subprocess, sys

def _pip(*args):
    subprocess.run([sys.executable, "-m", "pip", *args], check=False)

if BACKEND == "vllm":
    _pip("uninstall", "-y", "-q", "torchcodec")     # broken on Colab, breaks imports
    need_restart = False

    # Pin transformers + tokenizers FIRST (so vllm's deps don't override them),
    # then vllm. Detect whether anything changed -> restart needed.
    try:
        import transformers
        tf_ok = transformers.__version__.startswith("4.48")
    except Exception:
        tf_ok = False
    if not tf_ok:
        _pip("install", "-q", "transformers==4.48.3", "tokenizers==0.21.0"); need_restart = True
    else:
        print("transformers", transformers.__version__, "OK")

    try:
        import vllm
        vllm_ok = vllm.__version__.startswith("0.7")
    except Exception:
        vllm_ok = False
    if not vllm_ok:
        # --no-deps so vllm doesn't pull a newer transformers over our pin
        _pip("install", "-q", "vllm==0.7.3")
        _pip("install", "-q", "transformers==4.48.3", "tokenizers==0.21.0")  # re-pin after vllm
        need_restart = True
    else:
        print("vllm", vllm.__version__, "OK")

    if need_restart:
        print("\n>>> Installed pinned versions. RESTARTING runtime to apply them...")
        print(">>> After it restarts, run all cells again (Runtime -> Run all).")
        import os, time
        time.sleep(1)
        os.kill(os.getpid(), 9)     # Colab auto-restarts the kernel
    else:
        print("Dependencies ready (no restart needed).")
else:
    _pip("install", "-q", "-U", "accelerate")
    print("transformers backend — using Colab's native torch/transformers.")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(CACHE_DIR, exist_ok=True)
print("Drive mounted. CACHE_DIR =", CACHE_DIR)

## 4. Load the model + build `call_llm` (dynamic micro-batching)

A background batcher coalesces the concurrent per-claim requests into one
`generate([...])` call. With `vllm`, prefix caching reuses the shared few-shot
prompt prefix across all calls → big speedup. Keep `LLM_MAX_WORKERS ≥ TFM_BATCH`
so batches stay full.


In [ ]:
import threading, queue

_THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)
def _clean(text):
    if not text: return ""
    text = _THINK_RE.sub("", text)
    if "</think>" in text:
        text = text.split("</think>")[-1]
    return text.strip()

def _wrap_chat(user_msg, system_msg="You are a helpful assistant."):
    return (f"<|im_start|>system\n{system_msg}<|im_end|>\n"
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n")

_q = queue.Queue()

if BACKEND == "vllm":
    import transformers
    assert transformers.__version__.startswith("4.48"), (
        f"transformers is {transformers.__version__}, but vLLM 0.7.3 needs 4.48.x. "
        "Re-run the install cell and let it RESTART the runtime, then run all again."
    )
    from vllm import LLM, SamplingParams
    print(f"Loading {LLM_MODEL} with vLLM ...")
    _llm = LLM(
        model=LLM_MODEL,
        max_model_len=MAX_MODEL_LEN,
        gpu_memory_utilization=VLLM_GPU_UTIL,
        dtype="float16",
        enforce_eager=True,
        enable_prefix_caching=True,
        swap_space=16,                 # use the abundant free system RAM for KV overflow
        max_num_seqs=VLLM_MAX_NUM_SEQS,
        disable_log_stats=True,
    )
    _SP = SamplingParams(temperature=LLM_TEMPERATURE, max_tokens=LLM_MAX_TOKENS,
                         stop=["<|im_end|>", "<|im_start|>"])

    def _generate_batch(prompts):
        outs = _llm.generate([_wrap_chat(p) for p in prompts], _SP, use_tqdm=False)
        return [o.outputs[0].text for o in outs]

else:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    print(f"Loading {LLM_MODEL} with transformers (bf16) ...")
    _tok = AutoTokenizer.from_pretrained(LLM_MODEL)
    _tok.padding_side = "left"
    if _tok.pad_token is None:
        _tok.pad_token = _tok.eos_token
    _model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL, torch_dtype=torch.bfloat16).to("cuda").eval()

    def _generate_batch(prompts):
        texts = [_tok.apply_chat_template(
                    [{"role": "system", "content": "You are a helpful assistant."},
                     {"role": "user", "content": p}],
                    tokenize=False, add_generation_prompt=True) for p in prompts]
        enc = _tok(texts, return_tensors="pt", padding=True, truncation=True,
                   max_length=MAX_MODEL_LEN).to(_model.device)
        with torch.no_grad():
            out = _model.generate(**enc, max_new_tokens=LLM_MAX_TOKENS,
                                   do_sample=LLM_TEMPERATURE > 0,
                                   temperature=max(LLM_TEMPERATURE, 1e-2), top_p=0.9,
                                   pad_token_id=_tok.eos_token_id)
        gen = out[:, enc["input_ids"].shape[1]:]
        return _tok.batch_decode(gen, skip_special_tokens=True)

def _batch_worker():
    while True:
        first = _q.get()
        if first is None: return
        batch = [first]
        deadline = time.time() + TFM_COALESCE_WAIT
        while len(batch) < TFM_BATCH:
            try:
                nxt = _q.get(timeout=max(0.0, deadline - time.time()))
            except queue.Empty:
                break
            if nxt is None:
                _q.put(None); break
            batch.append(nxt)
        try:
            texts = _generate_batch([b["text"] for b in batch])
            for b, t in zip(batch, texts):
                b["result"] = _clean(t); b["event"].set()
        except Exception as e:
            print("[batcher] generate error:", e)
            for b in batch:
                b["result"] = ""; b["event"].set()

_batcher = threading.Thread(target=_batch_worker, daemon=True)
_batcher.start()

def call_llm(prompt, sweeps=1):
    item = {"text": prompt, "event": threading.Event(), "result": None}
    _q.put(item)
    item["event"].wait()
    return item["result"]

print(f"call_llm ready (backend={BACKEND}, batch={TFM_BATCH}, workers={LLM_MAX_WORKERS}).")
print("LLM test:", repr(call_llm("Reply with the single word: ok")[:50]))

## 5. Locate & load data (FactKG + DBpedia)

In [ ]:
def find_file(name):
    """Find a file by exact name under any of DATA_DIRS (first match wins)."""
    for root in DATA_DIRS:
        if not os.path.isdir(root):
            continue
        hits = glob.glob(os.path.join(root, "**", name), recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(
        f"Could not find {name}. Searched: {DATA_DIRS}. "
        f"Adjust DATA_DIRS in the Config cell to point at your KG-GPT data."
    )

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

FACTKG_PATHS = {
    "train": find_file("factkg_train.pickle"),
    "dev":   find_file("factkg_dev.pickle"),
    "test":  find_file("factkg_test.pickle"),
}
DBPEDIA_PATH   = find_file("dbpedia_2015_undirected_light.pickle")
RELATIONS_PATH = find_file("relations_for_final.pickle")
print("FactKG:", FACTKG_PATHS)
print("DBpedia:", DBPEDIA_PATH)
print("Relations:", RELATIONS_PATH)

print("\nLoading DBpedia (~1.5GB, may take a minute)...")
KG = load_pickle(DBPEDIA_PATH)
print("DBpedia entities:", len(KG))

## 6. Build `type_dict` (once, cached to Drive)

In [ ]:
from tqdm.auto import tqdm

TYPE_DICT_PATH = os.path.join(CACHE_DIR, "type_dict.pickle")

def build_type_dict(dbp, relations_path):
    relations = load_pickle(relations_path)
    relation_set = {}
    for rel in relations:
        if "~" not in rel[0]:
            relation_set[rel] = 0

    total = {}
    for ent in tqdm(list(dbp), desc="build type_dict"):
        try:
            bpl = dbp[ent]
            rels = list(bpl)
            head_types = bpl["22-rdf-syntax-ns#type"]
        except Exception:
            continue
        if len(rels) == 1 and "rdf-schema#label" in rels:
            continue
        for rel in rels:
            for tail in bpl[rel]:
                if '"' in tail:
                    continue
                try:
                    tail_types = dbp[tail]["22-rdf-syntax-ns#type"]
                    _ = relation_set[rel]
                except Exception:
                    continue
                for ht in head_types:
                    for tt in tail_types:
                        try:
                            total[ht][tt][rel] = 0
                        except Exception:
                            try:
                                total[ht][tt] = {}; total[ht][tt][rel] = 0
                            except Exception:
                                total[ht] = {}; total[ht][tt] = {}; total[ht][tt][rel] = 0
    return total

if os.path.exists(TYPE_DICT_PATH):
    print("Loading cached type_dict...")
    type_dict = load_pickle(TYPE_DICT_PATH)
else:
    type_dict = build_type_dict(KG, RELATIONS_PATH)
    with open(TYPE_DICT_PATH, "wb") as f:
        pickle.dump(type_dict, f)
print("type_dict head-types:", len(type_dict))

## 7. KG-GPT prompts + helper functions

In [ ]:
SENTENCE_DIVIDE_PROMPT = "Please divide the given sentence into several sentences each of which can be represented by one triplet. The generated sentences should be numbered and formatted as follows: #(number). (sentence), (entity set). The entity set for each sentence should contain no more than two entities, with each entity being used only once in all statements. The '##' symbol should be used to indicate an entity set. In the generated sentences, there cannot be more than two entities in the entity set. (i.e., the number of ## must not be larger than two.)\n\nExamples) \nSentence A: Ahmad Kadhim Assad's club is Al-Zawra'a SC.\nEntity set: ['Ahmad_Kadhim_Assad' ## \"Al-Zawra'a_SC\"] \n--> Divided: \n1. Ahmad Kadhim Assad's club is Al-Zawra'a SC., Entity set: ['Ahmad_Kadhim_Assad' ## \"Al-Zawra'a_SC\"] \n\n\nSentence B: Yeah! I know that Bananaman, which starred Tim Brooke-Taylor, first aired on 3rd October 1983!\nEntity set: ['\"1983-10-03\"' ## 'Tim_Brooke-Taylor' ## 'Bananaman'] \n--> Divided: \n1. Bananaman starred Tim Brooke-Taylor., Entity set: ['Bananaman' ## 'Tim_Brooke-Taylor'] \n2. Bananaman first aired on 3rd October 1983., Entity set: ['Bananaman' ## '\"1983-10-03\"']\n\n\nSentence C: AIDA Cruise line operated the ship which was built by Meyer Werft in Townsend, Poulshot, Wiltshire.\nEntity set: ['Meyer_Werft' ## 'AIDA_Cruises' ## '\"Townsend, Poulshot, Wiltshire\"']\n--> Divided: \n1. AIDA Cruise line operated the ship., Entity set: ['AIDA_Cruises' ## 'ship'] \n2. The ship was built by Meyer Werft., Entity set: ['ship' ## 'Meyer_Werft']\n3. Meyer Werft is located in Townsend, Poulshot, Wiltshire., Entity set: ['Meyer_Werft' ## '\"Townsend, Poulshot, Wiltshire\"']\n\n\nSentence D: Really? Jamie Lawrence is the music composer of the 83 minute 'Death on a Factory Farm' film, directed by Sarah Teale!\nEntity set: ['Jamie_Lawrence' ## '\"83.0\"' ## 'Death_on_a_Factory_Farm' ## 'Sarah_Teale'] \n--> Divided: \n1. Jamie Lawrence is the music composer of 'Death on a Factory Farm' film., Entity set: ['Jamie_Lawrence' ## 'Death_on_a_Factory_Farm'] \n2. 'Death on a Factory Farm' film is directed by Sarah Teale., Entity set: ['Death_on_a_Factory_Farm' ## 'Sarah_Teale']\n3. 'Death on a Factory Farm' is 83 minute film., Entity set: ['Death_on_a_Factory_Farm' ## '\"83.0\"']\n\n\nSentence E: Brandon Carter was born in England and graduated from the University of Cambridge where the current Chancellor is Leszek Borysiewicz.\nEntity set: ['Brandon_Carter' ## 'University_of_Cambridge' ## 'Leszek_Borysiewicz' ## 'England'] \n--> Divided: \n1. Brandon Carter was born in England., Entity set: ['Brandon_Carter' ## 'England'] \n2. Brandon Carter graduated from the University of Cambridge., Entity set: ['Brandon_Carter' ## 'University_of_Cambridge']\n3. The current Chancellor of University of Cambridge is Leszek Borysiewicz., Entity set: ['University_of_Cambridge' ## 'Leszek_Borysiewicz']\n\n\nSentence F: No, but the leader of the United States is not Olena Serdiuk.\nEntity set: ['United_States' ## '\"Olena Serdiuk\"'] \n--> Divided: \n1. The leader of the United States is not Olena Serdiuk., Entity set: ['United_States' ## '\"Olena Serdiuk\"'] \n\n\nSentence G: Yes, with a floor count of 45, 200 Public Square is located in Cleveland in the United States.\nEntity set: ['200_Public_Square' ## 'Cleveland' ## 'United_States' ## '\"45\"']\n--> Divided:\n1. 200 Public Square is with a floor count of 45., Entity set: ['200_Public_Square' ## '\"45\"']\n2. 200 Public Square is located in Cleveland., Entity set: ['200_Public_Square' ## 'Cleveland']\n3. Cleveland is in the United States., Entity set: ['Cleveland' ## 'United_States']\n\n\nSentence H: Bananaman the TV series starred by a person was shown on the company and the company headquarters is called Broadcasting House.\nEntity set: ['Broadcasting_House' ## 'Bananaman'] \n--> Divided: \n1. Bananaman the TV series starred by a person., Entity set: ['Bananaman' ## 'person'] \n2. Bananaman the TV series was shown on the company., Entity set: ['Bananaman' ## 'company']\n3. The company headquarters is called Broadcasting House., Entity set: ['company' ## 'Broadcasting_House']\n\n\nSentence I: Do you know Milan Hod\u017ea? he had a religion.\nEntity set: ['Milan_Hod\u017ea'] \n--> Divided: \n1. Milan Hod\u017ea had a religion., Entity set: ['Milan_Hod\u017ea'] \n\n\nSentence J: The place, designed by Huseyin Butuner and Hilmi Guner, is located in a country, where the leader is Artur Rasizade.\nEntity set: ['\"H\u00fcseyin B\u00fct\u00fcner and Hilmi G\u00fcner\"' ## 'Artur_Rasizade'] \n--> Divided: \n1. The place was designed by Huseyin Butuner and Hilmi Guner., Entity set: ['place' ## '\"H\u00fcseyin B\u00fct\u00fcner and Hilmi G\u00fcner\"'] \n2. The place is located in a country., Entity set: ['place' ## 'country']\n3. The leader of a country is Artur Rasizade., Entity set: ['country' ## 'Artur_Rasizade']\n\n\nSentence K: An academic journal with code IJPHDE is also Acta Math. Hungar.\nEntity set: ['\"Acta Math. Hungar.\"' ## '\"IJPHDE\"']\n--> Divided: \n1. An academic journal is with code IJPHDE., Entity set: ['academic journal' ## '\"IJPHDE\"']\n2. An academic journal is also Acta Math. Hungar., Entity set: ['academic journal' ## '\"Acta Math. Hungar.\"']\n\n\nSentence L: 'A film' was produced by Anatole de Grunwald, directed by Harold French, with cinematography done by Bernard Knowles.\nEntity set: ['Anatole_de_Grunwald' ## 'Bernard_Knowles' ## 'Harold_French'] \n--> Divided: \n1. 'A film' was produced by Anatole de Grunwald., Entity set: ['film' ## 'Anatole_de_Grunwald'] \n2. 'A film' was directed by Harold French., Entity set: ['film' ## ''Harold_French']\n3. 'A film' was with cinematography done by Bernard Knowles., Entity set: ['film' ## 'Bernard_Knowles']\n\n\nYour Task)\nSentence:  <<<<CLAIM>>>>\nEntity set: <<<<ENTITY_SET>>>>\n--> Divided:"

RELATION_RETRIEVAL_PROMPT = "I will give you a set of words. \nFind the top <<<<TOP_K>>>> elements from Words set which are most semantically related to the given sentence. You may select up to <<<<TOP_K>>>> words. If there is nothing that looks semantically related, pick out any <<<<TOP_K>>>> elements and give them to me.\n\nExamples)\nSentence A: Ahmad Kadhim Assad's club is Al-Zawra'a SC.\nWords set: ['club', 'clubs', 'parent', 'spouse', 'birthPlace', 'deathYear', 'leaderName', 'awards', 'award', 'vicepresident', 'vicePresident'] \nTop 2 Answer: ['club', 'clubs']\n\n\nSentence B: Yeah! I know that Bananaman, which starred Tim Brooke-Taylor, first aired on 3rd October 1983!\nWords set: ['OwningCompany', 'owner', 'dean', 'coach', 'writer', 'firstAired', 'director', 'formerTeam', 'starring'] \nTop 2 Answer: ['firstAired', 'starring'] \n\n\nSentence C: 'A film' was produced by Anatole de Grunwald, directed by Harold French, with cinematography done by Bernard Knowles.\nWords set: ['composer', 'team', 'editor', 'starring', 'runtime', 'director', 'discoverer', 'founder', 'crewMembers', 'writer', 'producer', 'cinematography'] \nTop 3 Answer: ['producer', 'director', 'cinematography']\n\n\nSentence D: Really? Jamie Lawrence is the music composer of the 83 minute 'Death on a Factory Farm' film, directed by Sarah Teale!\nWords set: ['composer', 'team', 'editor', 'starring', 'runtime', 'director', 'discoverer', 'founder', 'crewMembers', 'writer'] \nTop 4 Answer: ['composer', 'runtime', 'director', 'writer']\n\n\nSentence E: No, but the leader of the United States is not Olena Serdiuk.\nWords set: ['composer', 'team', 'editor', 'starring', 'runtime', 'director', 'discoverer', 'founder', 'leader', 'crewMembers', 'writer', 'producer', 'cinematography'] \nTop 1 Answer: ['leader']\n\n\nSentence F: Brandon Carter was born in England and graduated from the University of Cambridge where the current Chancellor is Leszek Borysiewicz.\nWords set: ['OwningCompany', 'placeOfBirth', 'owner', 'viceChancellor', 'almaMater', 'dean', 'coach', 'writer', 'firstAired', 'director', 'formerTeam', 'starring', 'birthPlace'] \nTop 4 Answer: ['birthPlace', 'placeOfBirth', 'viceChancellor', 'almaMater']\n\n\nSentence G: AIDA Cruise line operated the ship which was built by Meyer Werft in Townsend, Poulshot, Wiltshire.\nWords set: ['location', 'firstAired', 'clubs', 'parent', 'network', 'shipBuilder', 'birthPlace', 'locationCity', 'shipOperator', 'leaderName', 'awards', 'award', 'vicepresident', 'vicePresident'] \nTop 4 Answer: ['shipBuilder', 'location', 'locationCity', 'shipOperator',]\n\n\nSentence H: Yes, with a floor count of 45, 200 Public Square is located in Cleveland in the United States.\nWords set: ['producer', 'floorCount', 'country', 'location', 'primeMinister', 'parent', 'spouse', 'nativeName', 'builder', 'manager', 'designer'] \nTop 3 Answer: ['floorCount', 'country', 'location']\n\n\nSentence I: Bananaman the TV series starred by a person was shown on the company and the company headquarters is called Broadcasting House.\nWords set: ['composer', 'team', 'editor', 'starring', 'locationCity', 'runtime', 'network', 'discoverer', 'founder', 'crewMembers', 'writer'] \nTop 3 Answer: ['locationCity', 'starring','network']\n\n\nSentence J: Do you know Milan Hod\u017ea? he had a religion.\nWords set: ['deathYear', 'leaderName', 'awards', 'award', 'religion'] \nTop 1 Answer: ['religion']\n\n\nSentence K: The place, designed by Huseyin Butuner and Hilmi Guner, is located in a country, where the leader is Artur Rasizade.\nWords set: ['producer', 'primeMinister', 'parent', 'leaderName' 'spouse', 'nativeName', 'builder', 'manager', 'designer', 'location'] \nTop 3 Answer: ['designer', 'leaderName', 'location']\n\n\nSentence L: An academic journal with code IJPHDE is also Acta Math. Hungar.\nWords set: ['abbreviation', 'placeOfBirth', 'owner', 'coden', 'almaMater', 'dean', 'coach', 'writer', 'firstAired', 'director', 'formerTeam', 'starring', 'birthPlace'] \nTop 2 Answer: ['abbreviation', 'coden']\n\n\nNow let's find the top <<<<TOP_K>>>> elements.\nSentence: <<<<SENTENCE>>>>\nWords set: <<<<RELATION_SET>>>>\nTop <<<<TOP_K>>>> Answer: "

print("prompts embedded:", len(SENTENCE_DIVIDE_PROMPT), len(RELATION_RETRIEVAL_PROMPT))

In [ ]:
def claim_divider_parse_answer(answer, gt_entities):
    processed_answer_set = {}
    answer = (answer or "").strip()
    splitted_answers = answer.split("\n")
    all_entities = []
    try:
        for nth_answer in splitted_answers:
            nth_answer = nth_answer.strip()
            for i in range(3):
                if str(i + 1) + ". " in nth_answer[:5]:
                    temp_ans = nth_answer.split(str(i + 1) + ". ")[1]
                    temp_split = temp_ans.split(", Entity set: ")
                    sentence = temp_split[0]
                    entity_set = temp_split[1]
                    entity_set = entity_set.split("[")[1]
                    entity_set = entity_set.split("]")[0]
                    entity_set = entity_set.split(" ## ")
                    new_entity_set = []
                    for ent in entity_set:
                        new_entity_set.append(ent[1:-1]); all_entities.append(ent[1:-1])
                    break
            processed_answer_set[sentence] = new_entity_set
    except Exception:
        return None
    return processed_answer_set


def relation_candidates(KG, type_dict, entity_set):
    final_type_set = []; final_entity_set = []; new_entity_set = []
    for ent in entity_set:
        if '"' in ent:
            final_entity_set.append(ent); new_entity_set.append(ent); continue
        total = []; ent = ent.strip()
        splitted_ent = ent.split(" ")
        if len(splitted_ent) == 1:
            splitted_ent = ent.split("_")
        for spl_ent in splitted_ent:
            total.append([spl_ent.strip(), spl_ent.strip()[0].upper() + spl_ent.strip()[1:]])
        temp_list = []
        for chunk in total:
            if len(temp_list) == 0:
                temp_list = [chunk[0], chunk[1]]; continue
            new_list = []
            for temp in temp_list:
                new_list.append(temp + chunk[0]); new_list.append(temp + chunk[1])
            temp_list = new_list.copy()
        is_type = 0
        for type_ in temp_list:
            try:
                _ = type_dict[type_]; final_type_set.append([type_])
                new_entity_set.append(type_); is_type = 1; break
            except Exception:
                continue
        if is_type == 0:
            final_entity_set.append(ent); new_entity_set.append(ent)

    all_type_list = []
    if len(final_type_set) == 1:
        for temp_type in final_type_set[0]:
            for k in list(type_dict[temp_type]):
                all_type_list += type_dict[temp_type][k]
        all_type_list = list(set(all_type_list))
    else:
        for fs in final_type_set:
            tmp = []
            for ofs in final_type_set:
                if fs != ofs:
                    for fe in fs:
                        for oe in ofs:
                            if len(tmp) == 0:
                                try: tmp = type_dict[fe][oe]
                                except Exception: tmp = []
                            else:
                                try: tmp = [t for t in tmp if t in type_dict[fe][oe]]
                                except Exception: tmp = []
            all_type_list += tmp.copy()
        all_type_list = list(set(all_type_list))

    all_entity_list = []
    if len(final_entity_set) == 1:
        try: all_entity_list = list(KG[final_entity_set[0]])
        except Exception: all_entity_list = []
    else:
        for fe in final_entity_set:
            for ofe in final_entity_set:
                if fe != ofe:
                    try:
                        for temp_rel in list(KG[fe]):
                            try:
                                other = list(KG[ofe])
                                if "~" in temp_rel[0]:
                                    if temp_rel.split("~")[1] in other:
                                        all_entity_list.append(temp_rel.split("~")[1])
                                    elif "~" + temp_rel in other:
                                        all_entity_list.append(temp_rel)
                            except Exception: pass
                    except Exception: pass
        all_entity_list = list(set(all_entity_list))

    final_relation_list = []
    if len(all_type_list) == 0:
        for rel in all_entity_list:
            final_relation_list.append(rel.split("~")[1] if "~" in rel[0] else rel)
    elif len(all_entity_list) == 0:
        for rel in all_type_list:
            final_relation_list.append(rel.split("~")[1] if "~" in rel[0] else rel)
    else:
        for rel in all_entity_list:
            if "~" in rel[0]:
                if rel.split("~")[1] in all_type_list:
                    final_relation_list.append(rel.split("~")[1])
            elif rel in all_type_list or "~" + rel in all_type_list or len(all_type_list) == 0:
                final_relation_list.append(rel)
    return final_relation_list, new_entity_set


def retrieval_relation_parse_answer(answer):
    # Robust to chatty output: take the LAST bracketed list found.
    matches = re.findall(r"\[[^\]]+\]", answer or "")
    if not matches:
        return None
    match = matches[-1]
    comps = [c.strip() for c in match.strip("[]").split(",")]
    comps = [c.strip("''") if "'" in c else c for c in comps]
    comps = [c.strip().strip('"') for c in comps if c.strip()]
    return comps or None


def graph_extractor(target_list):
    if len(target_list) == 0:
        return target_list
    return_list = []; filter_dict = {"head": {}, "tail": {}}
    return_list.append(target_list[0])
    used_heads = []; used_tails = []
    filter_dict["head"][target_list[0][0]] = [target_list[0][1]]
    filter_dict["tail"][target_list[0][2]] = [target_list[0][1]]
    used_heads.append(target_list[0][0]); used_tails.append(target_list[0][2])
    for tar in target_list:
        if tar in return_list:
            continue
        try:
            if tar[1] in filter_dict["head"][tar[0]]:
                continue
        except Exception:
            try:
                if tar[1] in filter_dict["tail"][tar[2]]:
                    continue
            except Exception:
                pass
        try:
            if tar[1] not in list(filter_dict["head"][tar[0]]):
                filter_dict["head"][tar[0]] = [tar[1]]
                try: filter_dict["tail"][tar[2]].append(tar[1])
                except Exception: filter_dict["tail"][tar[2]] = [tar[1]]
                return_list.append(tar); continue
        except Exception:
            pass
        try:
            if tar[1] not in list(filter_dict["tail"][tar[2]]):
                filter_dict["tail"][tar[2]] = [tar[1]]
                try: filter_dict["head"][tar[0]].append(tar[1])
                except Exception: filter_dict["head"][tar[0]] = [tar[1]]
                return_list.append(tar); continue
        except Exception:
            pass
        if tar[2] in used_heads or tar[0] in used_tails:
            return_list.append(tar)
            try: filter_dict["head"][tar[0]].append(tar[1])
            except Exception: filter_dict["head"][tar[0]] = [tar[1]]
            try: filter_dict["tail"][tar[2]].append(tar[1])
            except Exception: filter_dict["tail"][tar[2]] = [tar[1]]
    return return_list

print("KG-GPT helpers ready.")

## 8. Stage 1+2 — adapter + triple pool builder

In [ ]:
def sentence_divide(claim, gt_entities):
    q = (SENTENCE_DIVIDE_PROMPT
         .replace("<<<<CLAIM>>>>", claim)
         .replace("<<<<ENTITY_SET>>>>", str(gt_entities)))
    for _ in range(3):
        out = call_llm(q)
        parsed = claim_divider_parse_answer(out, gt_entities)
        if parsed:
            return parsed
    return {claim: list(gt_entities)}   # fallback: whole claim as one sub-sentence


def top_k_relations(sub_sentence, candidates):
    if len(candidates) <= TOP_K:
        return list(candidates)
    q = (RELATION_RETRIEVAL_PROMPT
         .replace("<<<<TOP_K>>>>", str(TOP_K))
         .replace("<<<<SENTENCE>>>>", sub_sentence)
         .replace("<<<<RELATION_SET>>>>", str(candidates)))
    for _ in range(3):
        out = call_llm(q)
        picked = retrieval_relation_parse_answer(out)
        if picked:
            kept = [r for r in picked if r in candidates]
            return kept or list(candidates[:TOP_K])
    return list(candidates[:TOP_K])


def build_subclaim_triples(relations, entity_set):
    out = []
    for rel in relations:
        if len(entity_set) == 1:
            out.append([entity_set[0], rel]); out.append([entity_set[0], "~" + rel])
        for a in range(len(entity_set)):
            for b in range(len(entity_set)):
                if a != b:
                    out.append([entity_set[a], rel, entity_set[b]])
                    out.append([entity_set[a], "~" + rel, entity_set[b]])
    return out


def build_triple_pool(total_evidence, KG, type_dict, gt_entities, max_triples=MAX_TRIPLES):
    additional = []
    for evi in total_evidence:
        try:
            _ = type_dict[evi[0][0]]; _ = type_dict[evi[0][2]]
            for trip in evi: additional.append(trip[1])
        except Exception:
            continue

    before_final = []
    for evi in total_evidence:
        cur = []
        for trip in evi:
            try:
                _ = type_dict[trip[0]]; continue            # head is a type -> skip
            except Exception:
                try:
                    _ = type_dict[trip[2]]                    # tail is a type -> expand
                    try:
                        for tail in KG[trip[0]][trip[1]]:
                            cur.append([trip[0], trip[1], tail])
                    except Exception:
                        continue
                except Exception:
                    try:
                        if len(trip) == 2:
                            cur.append(trip)
                        elif trip[2] in KG[trip[0]][trip[1]]:
                            cur.append(trip)
                    except Exception:
                        pass
        if cur:
            before_final.append(cur)

    final_evidence = []
    for chunk in before_final:
        find_gt = 0
        for trip in chunk:
            if len(trip) != 2 and trip[0] in gt_entities and trip[2] in gt_entities:
                final_evidence.append(trip); find_gt = 1
        if find_gt == 1:
            continue
        if len(before_final) == 1:
            for trip in chunk:
                if len(trip) == 2:
                    try:
                        for tail in KG[trip[0]][trip[1]]:
                            final_evidence.append([trip[0], trip[1], tail])
                    except Exception:
                        continue
            break
        additional = list(set(additional))
        if len(additional) != 0:
            for sec in before_final:
                if chunk == sec: continue
                for trip in chunk:
                    if len(trip) == 2: continue
                    for st in sec:
                        if len(st) == 2: continue
                        for rel_ in additional:
                            for ti in [0, 2]:
                                for si in [0, 2]:
                                    for add in ["", "~"]:
                                        try:
                                            if add == "" and "~" in rel_:
                                                if trip[ti] in KG[st[si]][rel_.split("~")[1]]:
                                                    final_evidence += [trip, st, [st[si], rel_.split("~")[1], trip[ti]]]
                                        except Exception: pass
                                        try:
                                            if trip[ti] in KG[st[si]][add + rel_]:
                                                final_evidence += [trip, st, [st[si], add + rel_, trip[ti]]]
                                        except Exception: pass
        else:
            for sec in before_final:
                if chunk == sec: continue
                for trip in chunk:
                    for st in sec:
                        if len(trip) == 2 or len(st) == 2: continue
                        if (trip[0] in st and trip[0] not in gt_entities) or (trip[2] in st and trip[2] not in gt_entities):
                            final_evidence += [trip, st]

    new_final = []
    for trip in final_evidence:
        if "~" in trip[1]:
            flipped = [trip[2], trip[1].split("~")[1], trip[0]]
            if flipped not in new_final and flipped not in final_evidence:
                new_final.append(flipped); continue
        else:
            if trip not in new_final:
                new_final.append(trip)

    pruned = graph_extractor(new_final)
    pool, seen = [], set()
    for trip in pruned:
        if len(trip) >= 3:
            t = (trip[0], trip[1], trip[2])
            if t not in seen:
                seen.add(t); pool.append(t)
        if len(pool) >= max_triples:
            break
    return pool


def process_claim(claim, entity_set):
    divided = sentence_divide(claim, entity_set)

    # Leverage FactKG's gold Entity_set: the prompt already tells the LLM to use
    # only the given entities, but smaller models add junk (e.g. the pronoun
    # "he"). Keep only entities that are in entity_set (case-insensitive, mapped
    # back to the canonical KG key form). If a sub-sentence keeps nothing, drop
    # it; if everything is dropped, fall back to the whole claim + gold entities.
    _gold_lc = {e.lower(): e for e in entity_set}
    _cleaned = {}
    for _sub, _ents in divided.items():
        _kept = []
        for _e in _ents:
            if _e in entity_set:
                _kept.append(_e)
            elif _e.lower() in _gold_lc:
                _kept.append(_gold_lc[_e.lower()])   # canonicalize minor case diffs
        if _kept:
            _cleaned[_sub] = _kept
    divided = _cleaned if _cleaned else {claim: list(entity_set)}

    sub_data, total_evidence = [], []
    for sub_text, sub_entities in divided.items():
        try:
            cand, norm_ents = relation_candidates(KG, type_dict, sub_entities)
        except Exception:
            cand, norm_ents = [], list(sub_entities)
        if len(cand) == 0:
            chosen = []
        elif len(cand) < TOP_K:
            chosen = list(cand)
        else:
            chosen = top_k_relations(sub_text, cand)
        sub_data.append({"text": sub_text, "entities": list(norm_ents), "top_k_relations": list(chosen)})
        st = build_subclaim_triples(chosen, list(norm_ents))
        if st: total_evidence.append(st)
    pool = build_triple_pool(total_evidence, KG, type_dict, list(entity_set))
    return {"claim": claim, "entity_set": list(entity_set),
            "sub_sentences": sub_data, "triple_pool": pool}

print("Stage 1+2 ready.")

## 9. Run Stage 1+2 over each split (concurrent + resumable)

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

def _partial_path(split): return os.path.join(CACHE_DIR, f"stage12_{split}.partial.jsonl")
def _final_path(split):   return os.path.join(CACHE_DIR, f"stage12_{split}.pkl")

def _load_done(split):
    """qid -> record, seeded from a finished .pkl and/or the partial JSONL log."""
    done = {}
    fp = _final_path(split)
    if os.path.exists(fp):
        try:
            for r in load_pickle(fp):
                done[r["qid"]] = r
        except Exception:
            pass
    pp = _partial_path(split)
    if os.path.exists(pp):
        with open(pp, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    r = json.loads(line); done[r["qid"]] = r
                except Exception:
                    continue   # skip a half-written final line
    return done

def run_stage12(split, limit):
    if limit == 0:
        print(f"[{split}] skipped (limit=0)")
        return None

    factkg = load_pickle(FACTKG_PATHS[split])
    items = list(factkg.items())
    if limit and limit > 0:
        items = items[:limit]

    done = _load_done(split)
    todo = [(i, it) for i, it in enumerate(items) if i not in done]
    print(f"[{split}] total={len(items)} done={len(done)} todo={len(todo)}")

    def work(args):
        qid, (claim_text, cdata) = args
        try:
            out = process_claim(claim_text, cdata.get("Entity_set", []))
            lab = cdata.get("Label", [None])
            out["qid"] = qid
            out["label"] = lab[0] if isinstance(lab, (list, tuple)) and lab else lab
            out["reasoning_types"] = cdata.get("types", [])
            return out
        except Exception as e:
            print(f"[{split}] qid={qid} error: {e}")
            return None

    if todo:
        lock = threading.Lock()
        with open(_partial_path(split), "a", encoding="utf-8") as jf, \
             ThreadPoolExecutor(max_workers=LLM_MAX_WORKERS) as ex:
            futs = [ex.submit(work, t) for t in todo]
            for fut in tqdm(as_completed(futs), total=len(futs), desc=f"Stage12::{split}"):
                r = fut.result()
                if r is None:
                    continue
                with lock:                              # durable per-claim checkpoint
                    jf.write(json.dumps(r, ensure_ascii=False) + "\n"); jf.flush()
                    done[r["qid"]] = r

    records = [done[q] for q in sorted(done)]
    with open(_final_path(split), "wb") as f:
        pickle.dump(records, f)

    sizes = [len(r["triple_pool"]) for r in records]
    empty = sum(1 for s in sizes if s == 0)
    avg = (sum(sizes) / len(sizes)) if sizes else 0
    print(f"[{split}] saved {len(records)} -> {_final_path(split)} | "
          f"avg pool={avg:.2f}, empty={empty} ({100*empty/max(len(records),1):.1f}%)")
    return _final_path(split)

train_cache = run_stage12("train", TRAIN_LIMIT)
dev_cache   = run_stage12("dev",   DEV_LIMIT)
test_cache  = run_stage12("test",  TEST_LIMIT)

## 9b. (Optional) Re-process only the empty-pool claims
Keeps the cache; re-runs `process_claim` only on records with an empty `triple_pool` (e.g. after the gold-entity fix). Marks tried ones so a re-run won't redo genuine empties; keeps the resume log in sync.

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor, as_completed

def fix_empty_pools(cache_path, save_every=2000):
    if not os.path.exists(cache_path):
        print(cache_path, "-> not found, skip"); return
    with open(cache_path, "rb") as f:
        records = pickle.load(f)
    todo = [i for i, r in enumerate(records)
            if len(r["triple_pool"]) == 0 and not r.get("_tried_fix")]
    print(f"{os.path.basename(cache_path)}: empty&untried={len(todo)} / total={len(records)}")
    if not todo:
        print("  nothing to do."); return

    def work(i):
        r = records[i]
        try:
            out = process_claim(r["claim"], r["entity_set"])
            return i, out["triple_pool"], out["sub_sentences"]
        except Exception as e:
            print(f"  qid={r.get('qid')} error: {e}")
            return i, [], None

    def _save():
        with open(cache_path, "wb") as f:
            pickle.dump(records, f)
        pp = cache_path.replace(".pkl", ".partial.jsonl")     # keep resume log in sync
        with open(pp, "w", encoding="utf-8") as f:
            for r in records:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")

    done = filled = 0
    with ThreadPoolExecutor(max_workers=LLM_MAX_WORKERS) as ex:
        futs = [ex.submit(work, i) for i in todo]
        for fut in tqdm(as_completed(futs), total=len(futs), desc="fix empties"):
            i, pool, subs = fut.result()
            records[i]["triple_pool"] = pool
            if subs is not None:
                records[i]["sub_sentences"] = subs
            records[i]["_tried_fix"] = True
            done += 1
            if len(pool) > 0: filled += 1
            if done % save_every == 0: _save()
    _save()
    still = sum(1 for r in records if len(r["triple_pool"]) == 0)
    print(f"  newly filled={filled}, still empty={still} ({100*still/len(records):.1f}%)")

# Run on whichever caches exist (test first to gauge, then train).
for _split in ["test", "dev", "train"]:
    fix_empty_pools(os.path.join(CACHE_DIR, f"stage12_{_split}.pkl"))

## 10. Preview the cached output

In [ ]:
def preview(cache_path, k=3):
    if not cache_path or not os.path.exists(cache_path):
        print(cache_path, "-> (skipped / not found)"); return
    with open(cache_path, "rb") as f:
        recs = pickle.load(f)
    sizes = [len(r["triple_pool"]) for r in recs]
    empty = sum(1 for s in sizes if s == 0)
    avg = (sum(sizes)/len(sizes)) if sizes else 0
    print(f"\n=== {os.path.basename(cache_path)} ===")
    print(f"records={len(recs)} | avg pool={avg:.2f} | empty={empty} "
          f"({100*empty/max(len(recs),1):.1f}%)")
    for r in recs[:k]:
        print("-"*60)
        print("claim :", r["claim"][:90])
        print("label :", r["label"], "| types:", r.get("reasoning_types"))
        print("pool  :", r["triple_pool"][:5], ("..." if len(r["triple_pool"])>5 else ""))

for p in [train_cache, dev_cache, test_cache]:
    preview(p)

## 11. Notes

- **Speed**: raise `TFM_BATCH` / `VLLM_MAX_NUM_SEQS` (A100 has room) and keep
  `LLM_MAX_WORKERS ≥ TFM_BATCH`. Prefix caching makes the shared few-shot prompt
  nearly free after the first call.
- **Resume**: cache lives in `MyDrive/kernelkg_gpt/cache/`. If Colab
  disconnects, re-run — it continues from `stage12_{split}.partial.jsonl`.
- **type_dict** is built once over DBpedia and cached to Drive.
- **Next**: feed `stage12_{train,dev,test}.pkl` into the Stage-3 training
  notebook (`KernelKG_GPT_Kaggle.ipynb`).
- If `vllm` ever fails to install/load, set `BACKEND="transformers"` in Config
  and re-run from there.
